# 6 · Numbers in the main text

This page computes the additional numbers quoted in the main text from
the prepared student and dyad datasets. Its calculations include the
all-dyad model needed for the quartile comparison, so no saved Table 1
is required.

**Sample and design.** Size of the sample, number of decisions per child
and rounds per classroom.

In [ ]:
here::i_am("notebooks/main/08_numbers_in_text.qmd")
library(tidyverse)
library(lfe)
library(igraph)
library(here)

The data are reused if `analysis_dyads` and `analysis_students` are
already in R. Otherwise, this step loads their CSV files from
`data/generated/main/`, or builds them from the raw inputs on first use.

In [ ]:
source(here("code", "00_load_main_data.R"), local = TRUE)
dyads <- analysis_dyads %>%
  mutate(class_round = paste(class_id, round, sep = "_"))
students <- analysis_students

numbers <- list()

In [ ]:
main_data_status

dataset,rows,source
analysis_dyads,17804,Saved CSV files
analysis_students,859,Saved CSV files


In [ ]:
# Abstract, Results and Methods: who took part
numbers$children   <- nrow(students)
numbers$classrooms <- n_distinct(students$class_id)
numbers$schools    <- n_distinct(students$school_id)
numbers$decisions  <- nrow(dyads)
numbers$percent_female <- 100 * mean(students$female)
# two computed ages above 15 are birth-date errors and are left out (as in the SI)
valid_age <- students$age[!is.na(students$age) & students$age <= 15]
numbers$age_mean   <- mean(valid_age)
numbers$age_sd     <- sd(valid_age)

numbers[c("children", "classrooms", "schools", "decisions",
          "percent_female", "age_mean", "age_sd")]

In [ ]:
numbers[c("children", "classrooms", "schools", "decisions", "percent_female", "age_mean", "age_sd")]
## $children
## [1] 859
## 
## $classrooms
## [1] 45
## 
## $schools
## [1] 14
## 
## $decisions
## [1] 17804
## 
## $percent_female
## [1] 57.2759
## 
## $age_mean
## [1] 10.05088
## 
## $age_sd
## [1] 1.01552

In [ ]:
# Methods, "Random assignment of dyadic partners"
decisions_per_child <- dyads %>% count(sender_id)
rounds_per_class    <- dyads %>% group_by(class_id) %>% summarise(rounds = n_distinct(round))

numbers$decisions_per_child_min  <- min(decisions_per_child$n)
numbers$decisions_per_child_max  <- max(decisions_per_child$n)
numbers$decisions_per_child_mean <- mean(decisions_per_child$n)
numbers$rounds_per_class_min     <- min(rounds_per_class$rounds)
numbers$rounds_per_class_max     <- max(rounds_per_class$rounds)
numbers$rounds_per_class_mean    <- mean(rounds_per_class$rounds)

# Each pair of classmates met only once: count unordered pairs and check that
# no pair appears in more than one round
pairs <- dyads %>%
  mutate(child_a = pmin(sender_id, receiver_id),
         child_b = pmax(sender_id, receiver_id)) %>%
  group_by(child_a, child_b) %>%
  summarise(decisions = n(), rounds = n_distinct(round), .groups = "drop")

numbers$unordered_pairs        <- nrow(pairs)
numbers$pairs_in_several_rounds <- sum(pairs$rounds > 1)
numbers$pairs_with_two_decisions <- sum(pairs$decisions == 2)

numbers[str_detect(names(numbers), "per_|pairs")]

In [ ]:
numbers[grepl("per_|pairs", names(numbers))]
## $decisions_per_child_min
## [1] 6
## 
## $decisions_per_child_max
## [1] 38
## 
## $decisions_per_child_mean
## [1] 20.72643
## 
## $rounds_per_class_min
## [1] 7
## 
## $rounds_per_class_max
## [1] 43
## 
## $rounds_per_class_mean
## [1] 23.93333
## 
## $unordered_pairs
## [1] 8902
## 
## $pairs_in_several_rounds
## [1] 0
## 
## $pairs_with_two_decisions
## [1] 8902

**Friendship.** Share of non-friend, mutual and one-sided pairs.

In [ ]:
# Results and Fig. 3 legend: share of each relationship type
friendship_shares <- dyads %>%
  mutate(type = if_else(friendship %in% c("Sender declares", "Receiver declares"),
                        "Asymmetric", friendship)) %>%
  count(type) %>%
  mutate(percent = 100 * n / sum(n))

friendship_shares

numbers$percent_non_friends <- friendship_shares$percent[friendship_shares$type == "Non-friends"]
numbers$percent_mutual      <- friendship_shares$percent[friendship_shares$type == "Mutual"]
numbers$percent_asymmetric  <- friendship_shares$percent[friendship_shares$type == "Asymmetric"]

# Results, "Friendship modulates...": spread of the rank gap in each group
numbers$sd_rank_gap_mutual      <- sd(dyads$rank_gap[dyads$friendship == "Mutual"])
numbers$sd_rank_gap_non_friends <- sd(dyads$rank_gap[dyads$friendship == "Non-friends"])

In [ ]:
friendship_shares

type,n,percent
Asymmetric,2600,14.603460
Mutual,1288,7.234329
Non-friends,13916,78.162211


**Rank differences children actually meet.** The full range of the rank
gap (from bottom to top of a classroom) occurs for only one pair per
classroom. This block looks at a more common contrast: pairs between a
child in the top quarter of the classroom ranking and one in the bottom
quarter, and the Cooperation GAP that Table 1 predicts for them.

In [ ]:
# Results: "a top- versus bottom-quartile pairing, about 13% of interactions,
# implies roughly 1.8 tokens".
# Quartiles of Social Rank are computed over all senders and receivers.
quartile_cuts <- quantile(c(dyads$sender_rank, dyads$receiver_rank), c(0.25, 0.50, 0.75))

to_quartile <- function(rank) {
  case_when(rank <= quartile_cuts[1] ~ 1,
            rank <= quartile_cuts[2] ~ 2,
            rank <= quartile_cuts[3] ~ 3,
            TRUE ~ 4)
}

dyads <- dyads %>%
  mutate(sender_quartile   = to_quartile(sender_rank),
         receiver_quartile = to_quartile(receiver_rank))

# Fit the all-dyad specification directly for the quartile prediction.
all_dyad_model <- felm(coop_gap ~ rank_gap | sender_id + class_round | 0 | class_id,
                      data = dyads)
beta_all <- coef(all_dyad_model)[["rank_gap"]]

top_to_bottom <- dyads %>% filter(sender_quartile == 4, receiver_quartile == 1)

# share of pairs between the top and the bottom quartile (in either direction),
# computed in each classroom and then averaged over the 45 classrooms
numbers$percent_top_vs_bottom_pairs <- dyads %>%
  group_by(class_id) %>%
  summarise(share = mean(abs(sender_quartile - receiver_quartile) == 3)) %>%
  summarise(100 * mean(share)) %>%
  pull()
numbers$predicted_coop_gap_top_to_bottom <- beta_all * mean(top_to_bottom$rank_gap)
numbers$observed_coop_gap_top_to_bottom  <- mean(top_to_bottom$coop_gap)

In [ ]:
numbers[c("percent_top_vs_bottom_pairs", "predicted_coop_gap_top_to_bottom", "observed_coop_gap_top_to_bottom")]
## $percent_top_vs_bottom_pairs
## [1] 13.29799
## 
## $predicted_coop_gap_top_to_bottom
## [1] 1.758509
## 
## $observed_coop_gap_top_to_bottom
## [1] 1.541916

**Filtered analysis.** The Table 1 model uses `coop_gap_filtered`, which
is observed only when both directed transfers pass the filter. Figure 3C
summarizes retained transfers within children instead; its filtered
curve and these regressions use different aggregation rules.

In [ ]:
# Results and Methods: null-model filtered regressions (Fig. 3C, SI tables)
numbers$percent_transfers_kept <- 100 * mean(!is.na(dyads$tokens_sent_filtered))
numbers$percent_dyads_kept     <- 100 * mean(!is.na(dyads$coop_gap_filtered))

filtered_slope <- function(data) {
  model <- felm(coop_gap_filtered ~ rank_gap | sender_id + class_round | 0 | class_id,
                data = data %>% filter(!is.na(coop_gap_filtered)))
  summary(model)$coefficients["rank_gap", ]
}

filtered_all <- filtered_slope(dyads)
filtered_non <- filtered_slope(dyads %>% filter(friendship == "Non-friends"))
filtered_mut <- filtered_slope(dyads %>% filter(friendship == "Mutual"))

rbind(all = filtered_all, non_friends = filtered_non, mutual = filtered_mut)

numbers$filtered_slope_all         <- filtered_all[["Estimate"]]
numbers$filtered_slope_non_friends <- filtered_non[["Estimate"]]
numbers$filtered_p_non_friends     <- filtered_non[["Pr(>|t|)"]]
numbers$filtered_slope_mutual      <- filtered_mut[["Estimate"]]

In [ ]:
rbind(all = filtered_all, non_friends = filtered_non, mutual = filtered_mut)
##             Estimate Cluster s.e.  t value     Pr(>|t|)
## all         3.445687    0.5094643 6.763352 2.543135e-08
## non_friends 3.406916    0.7250143 4.699102 2.586948e-05
## mutual      5.199171    3.4634512 1.501153 1.449210e-01

**One-sided friendships.** How often the exchanges fall in each region
of Figure 3A.

In [ ]:
# Results, "Directional effects in asymmetric friendships": one-sided pairs
asymmetric <- dyads %>% filter(friendship == "Sender declares")   # each pair once

numbers$percent_full_cooperation_asym <-
  100 * mean(asymmetric$tokens_sent == 10 & asymmetric$tokens_received == 10)
# "one student sent at least five tokens more than they received"
numbers$percent_strongly_asymmetric_asym <-
  100 * mean(abs(asymmetric$tokens_sent - asymmetric$tokens_received) >= 5)
# the blue region of the heatmap (one sent 10, the other 5 or less)
numbers$percent_blue_region_asym <-
  100 * mean(pmax(asymmetric$tokens_sent, asymmetric$tokens_received) == 10 &
             pmin(asymmetric$tokens_sent, asymmetric$tokens_received) <= 5)

In [ ]:
numbers[c("percent_full_cooperation_asym", "percent_strongly_asymmetric_asym", "percent_blue_region_asym")]
## $percent_full_cooperation_asym
## [1] 14.61538
## 
## $percent_strongly_asymmetric_asym
## [1] 36.76923
## 
## $percent_blue_region_asym
## [1] 28.53846

**A rank read off the game.** Instead of nominations, rank can be
computed from the token network itself (PageRank on who gives to whom).
This block correlates that behavioral rank with the nomination-based
Social Rank.

In [ ]:
# Discussion: Behavioral Social Rank = PageRank on the network of tokens sent
# in the game, rescaled 0-1 within class. Compared with the nomination-based
# Social Rank using a Spearman correlation.
behavioral_rank <- dyads %>%
  transmute(class_id, from = as.character(sender_id), to = as.character(receiver_id),
            weight = tokens_sent) %>%
  group_by(class_id) %>%
  group_modify(function(edges, key) {
    pr <- page_rank(graph_from_data_frame(edges, directed = TRUE))$vector
    tibble(student_id = as.numeric(names(pr)), behavioral_pagerank = pr)
  }) %>%
  mutate(behavioral_rank = (behavioral_pagerank - min(behavioral_pagerank)) /
                           (max(behavioral_pagerank) - min(behavioral_pagerank))) %>%
  ungroup() %>%
  left_join(students %>% select(student_id, social_rank), by = "student_id")

# Three versions of the correlation: (1) raw PageRank scores pooled over all
# classrooms; (2) the 0-1 ranks rescaled within each classroom, pooled; and
# (3) the median of the within-classroom correlations. Raw PageRank is
# roughly 1 / class size, so (1) also reflects differences in class size;
# (2) and (3) compare children only within their own classroom.
numbers$spearman_raw_pagerank_pooled <-
  cor(behavioral_rank$behavioral_pagerank,
      students$pagerank[match(behavioral_rank$student_id, students$student_id)],
      method = "spearman")
numbers$spearman_rescaled_ranks_pooled <-
  cor(behavioral_rank$behavioral_rank, behavioral_rank$social_rank, method = "spearman")
numbers$spearman_median_within_class <- behavioral_rank %>%
  group_by(class_id) %>%
  summarise(rho = cor(behavioral_rank, social_rank, method = "spearman")) %>%
  summarise(median(rho)) %>%
  pull()

In [ ]:
numbers[grepl("spearman", names(numbers))]
## $spearman_raw_pagerank_pooled
## [1] 0.8522072
## 
## $spearman_rescaled_ranks_pooled
## [1] 0.4311976
## 
## $spearman_median_within_class
## [1] 0.5093583

**Response quality.** Children who named the same classmate as both
popular and not popular, and the time between consecutive decisions.

In [ ]:
# Methods, response quality.
# (1) Children who named the same classmate as both "most popular" and
#     "not popular" (questions 4 and 5).
conflicting <- dyads %>% filter(nom_q4 == 1 & nom_q5 == 1)

numbers$conflicting_popularity_pairs    <- nrow(conflicting)                 # SI: 121
numbers$children_with_conflict          <- n_distinct(conflicting$sender_id) # SI: 70 of 859
numbers$classes_with_conflict           <- n_distinct(conflicting$class_id)  # SI: 30 of 45
# each conflict involves two nominations (one in Q4, one in Q5)
numbers$percent_of_popularity_nominations <- 100 * 2 * nrow(conflicting) /
  (sum(dyads$nom_q4) + sum(dyads$nom_q5))

# (2) Time between consecutive decisions of the same child (only classes
#     where the tablets recorded the time of each decision).
intervals <- dyads %>%
  filter(!is.na(seconds_since_start)) %>%
  arrange(sender_id, seconds_since_start) %>%
  group_by(sender_id) %>%
  mutate(seconds_since_previous = seconds_since_start - lag(seconds_since_start)) %>%
  ungroup()

numbers$median_seconds_between_decisions <- median(intervals$seconds_since_previous, na.rm = TRUE)
numbers$classes_with_timing <- n_distinct(intervals$class_id)

In [ ]:
numbers[c("conflicting_popularity_pairs", "children_with_conflict", "classes_with_conflict", "percent_of_popularity_nominations", "median_seconds_between_decisions", "classes_with_timing")]
## $conflicting_popularity_pairs
## [1] 121
## 
## $children_with_conflict
## [1] 70
## 
## $classes_with_conflict
## [1] 30
## 
## $percent_of_popularity_nominations
## [1] 5.792245
## 
## $median_seconds_between_decisions
## [1] 44
## 
## $classes_with_timing
## [1] 29

In [ ]:
numbers_table <- tibble(quantity = names(numbers), value = unlist(numbers))
numbers_table %>% mutate(value = round(value, 3)) %>% print(n = Inf)

write_csv(numbers_table, here("output", "tables", "numbers_in_text.csv"))

In [ ]:
numbers_table

quantity,value
children,8.590000e+02
classrooms,4.500000e+01
schools,1.400000e+01
decisions,1.780400e+04
percent_female,5.727590e+01
age_mean,1.005088e+01
age_sd,1.015520e+00
decisions_per_child_min,6.000000e+00
decisions_per_child_max,3.800000e+01
decisions_per_child_mean,2.072643e+01


# Session information

In [ ]:
sessionInfo()
## R version 4.3.2 (2023-10-31)
## Platform: aarch64-apple-darwin20 (64-bit)
## Running under: macOS 26.7.1
## 
## Matrix products: default
## BLAS:   /Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/lib/libRblas.0.dylib 
## LAPACK: /Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/lib/libRlapack.dylib;  LAPACK version 3.11.0
## 
## locale:
## [1] C.UTF-8/C.UTF-8/C.UTF-8/C/C.UTF-8/C.UTF-8
## 
## time zone: America/Santiago
## tzcode source: internal
## 
## attached base packages:
## [1] stats     graphics  grDevices utils     datasets  methods   base     
## 
## other attached packages:
##  [1] igraph_2.1.4    lfe_3.1.1       Matrix_1.6-1.1  lubridate_1.9.4
##  [5] forcats_1.0.0   stringr_1.5.1   dplyr_1.1.4     purrr_1.0.4    
##  [9] readr_2.1.5     tidyr_1.3.1     tibble_3.2.1    ggplot2_3.5.2  
## [13] tidyverse_2.0.0 here_1.0.1     
## 
## loaded via a namespace (and not attached):
##  [1] utf8_1.2.4        sandwich_3.1-1    generics_0.1.3    stringi_1.8.7    
##  [5] lattice_0.22-6    hms_1.1.3         digest_0.6.37     magrittr_2.0.3   
##  [9] evaluate_1.0.3    grid_4.3.2        timechange_0.3.0  fastmap_1.2.0    
## [13] rprojroot_2.0.4   jsonlite_2.0.0    Formula_1.2-5     scales_1.3.0     
## [17] cli_3.6.4         crayon_1.5.3      rlang_1.1.6       bit64_4.6.0-1    
## [21] munsell_0.5.1     withr_3.0.2       yaml_2.3.10       tools_4.3.2      
## [25] parallel_4.3.2    tzdb_0.5.0        colorspace_2.1-1  vctrs_0.6.5      
## [29] R6_2.6.1          zoo_1.8-13        lifecycle_1.0.4   bit_4.6.0        
## [33] vroom_1.6.5       pkgconfig_2.0.3   pillar_1.10.2     gtable_0.3.6     
## [37] glue_1.8.0        xfun_0.52         tidyselect_1.2.1  knitr_1.50       
## [41] xtable_1.8-4      htmltools_0.5.8.1 rmarkdown_2.29    compiler_4.3.2